In [1]:
!pip install -q transformers accelerate

In [3]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get("hf_token"))

In [4]:
import torch
from transformers import AutoProcessor, Gemma3ForConditionalGeneration

model_id = "google/gemma-3-4b-it"

model = Gemma3ForConditionalGeneration.from_pretrained(
    model_id, device_map="auto", torch_dtype=torch.bfloat16
).eval()

processor = AutoProcessor.from_pretrained(model_id)

print(f"Model loaded — dtype: {next(model.parameters()).dtype}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

Model loaded — dtype: torch.bfloat16


In [5]:
from google.colab import files
from PIL import Image
import torch

uploaded = files.upload()
filename = list(uploaded.keys())[0]
image = Image.open(filename).convert("RGB")

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": "What animal is this?"},
        ]
    }
]

inputs = processor.apply_chat_template(
    messages, add_generation_prompt=True, tokenize=True,
    return_dict=True, return_tensors="pt"
).to(model.device, dtype=torch.bfloat16)

input_len = inputs["input_ids"].shape[-1]

with torch.inference_mode():
    generation = model.generate(**inputs, max_new_tokens=200, do_sample=False)
    generation = generation[0][input_len:]

print(processor.decode(generation, skip_special_tokens=True))

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Saving Bengal-Tiger-wikipedia.jpg to Bengal-Tiger-wikipedia.jpg
Based on the image, this is a **tiger**. 

Specifically, it appears to be a **Bengal tiger** due to its coloration and markings – the orange coat with black stripes. 

Do you want to know anything more about tigers, like their habitat, behavior, or conservation status?


In [6]:
baseline_stats = {}
for name, module in model.named_modules():
    if isinstance(module, torch.nn.Linear):
        w = module.weight.data.float()
        flat = w.reshape(-1)  # Flatten entire tensor, not into blocks
        scale = w.abs().amax() / 127.0  # Per-tensor scale, not per-block
        fractional = (flat / scale - (flat / scale).round()).abs()
        baseline_stats[name] = {
            "unique_vals": flat.unique().numel(),
            "mean_fractional": fractional.mean().item(),
        }

print(f"Baseline stats captured for {len(baseline_stats)} Linear layers ✓")

Baseline stats captured for 401 Linear layers ✓


In [7]:
import torch

def uniform_quantize_simple(tensor, bits=2):
    """Per-tensor symmetric quantization (simplest baseline)"""
    max_val = 2 ** (bits - 1) - 1
    scale = tensor.abs().amax() / max_val
    scale = scale.clamp(min=1e-8)

    quantized = (tensor / scale).round().clamp(-max_val, max_val)
    dequantized = quantized * scale

    return dequantized


# Apply INT2 quantization
count = 0
with torch.no_grad():
    for name, module in model.named_modules():
        if isinstance(module, torch.nn.Linear):
            module.weight.data = uniform_quantize_simple(module.weight.data, bits=2)
            count += 1

print(f"INT2 quantization applied to {count} Linear layers")

INT2 quantization applied to 401 Linear layers


In [8]:
results = []
for name, module in model.named_modules():
    if isinstance(module, torch.nn.Linear):
        flat = module.weight.data.float().cpu().reshape(-1)  # Flatten entire tensor
        scale = flat.abs().amax() / 1.0  # Per-tensor scale for INT2
        fractional = (flat / scale - (flat / scale).round()).abs()
        unique_vals = flat.unique().numel()  # Unique vals across entire tensor
        results.append((name, fractional.mean().item(), unique_vals))

print(f"Total Linear layers quantized: {len(results)}\n")

for idx in [0, len(results)//2, len(results)-1]:
    name, frac, uniq = results[idx]
    if baseline_stats:
        baseline_frac = baseline_stats[name]["mean_fractional"]
        baseline_uniq = baseline_stats[name]["unique_vals"]
        print(f"Layer: {name}")
        print(f"  Fractional (base → quant): {baseline_frac:.8f} → {frac:.8f}")
        print(f"  Unique vals (base → quant): {baseline_uniq} → {uniq}")
    else:
        print(f"Layer: {name}")
        print(f"  Fractional (quant): {frac:.8f}")
        print(f"  Unique vals (quant): {uniq}")

print(f"\nMax fractional across all layers: {max(f for _,f,_ in results):.8f}")
print(f"All layers ≤256 unique vals: {all(u <= 256 for _,_,u in results)}")

Total Linear layers quantized: 401

Layer: model.vision_tower.vision_model.encoder.layers.0.self_attn.k_proj
  Fractional (base → quant): 0.24999444 → 0.00000000
  Unique vals (base → quant): 4471 → 3
Layer: model.language_model.layers.5.self_attn.o_proj
  Fractional (base → quant): 0.24983278 → 0.00000000
  Unique vals (base → quant): 4262 → 3
Layer: lm_head
  Fractional (base → quant): 0.25035596 → 0.00000000
  Unique vals (base → quant): 6539 → 3

Max fractional across all layers: 0.00000000
All layers ≤256 unique vals: True


In [10]:
from PIL import Image
import torch

image = Image.open("Bengal-Tiger-wikipedia.jpg").convert("RGB")

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": "What animal is this?"},
        ]
    }
]

inputs = processor.apply_chat_template(
    messages, add_generation_prompt=True, tokenize=True,
    return_dict=True, return_tensors="pt"
).to(model.device, dtype=torch.bfloat16)

input_len = inputs["input_ids"].shape[-1]

with torch.inference_mode():
    generation = model.generate(**inputs, max_new_tokens=200, do_sample=False)
    generation = generation[0][input_len:]

print("INT2 Output:", processor.decode(generation, skip_special_tokens=True))

INT2 Output: खनऊखनऊ


In [11]:
from google.colab import drive
drive.mount('/content/drive')

import json
import base64
from PIL import Image
import io
from tqdm import tqdm

DRIVE_FILE = "/content/drive/MyDrive/vqav2_5k.jsonl"

samples = []
with open(DRIVE_FILE, "r") as f:
    for line in tqdm(f, total=5000):
        entry = json.loads(line)
        img_bytes = base64.b64decode(entry["image_b64"])
        img = Image.open(io.BytesIO(img_bytes)).convert("RGB")
        samples.append({
            "question_id": entry["question_id"],
            "question": entry["question"],
            "image": img,
            "answers": entry["answers"],
        })

print(f"Loaded {len(samples)} samples ✓")

Mounted at /content/drive


100%|██████████| 5000/5000 [00:16<00:00, 308.05it/s]

Loaded 5000 samples ✓


In [12]:
import time

batch = samples[:32]

batch_messages = [
    [{"role": "user", "content": [
        {"type": "image", "image": s["image"]},
        {"type": "text", "text": s["question"] + " Answer the question using a single word or phrase."},
    ]}]
    for s in batch
]

inputs = processor.apply_chat_template(
    batch_messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt",
    padding=True,
)
inputs = inputs.to(model.device)

start = time.time()
with torch.no_grad():
    print("Input shape:", inputs["input_ids"].shape)
    print("VRAM used (GB):", torch.cuda.memory_allocated() / 1e9)
    generated_ids = model.generate(**inputs, max_new_tokens=32)
elapsed = time.time() - start

input_len = inputs["input_ids"].shape[1]
preds = processor.batch_decode(
    generated_ids[:, input_len:],
    skip_special_tokens=True,
    clean_up_tokenization_spaces=False
)

sps = 32 / elapsed
print(f"32 samples in {elapsed:.1f}s — {sps:.2f} samples/sec — ETA for 5k: {5000/sps/60:.1f} mins\n")

for s, pred in zip(batch[:5], preds[:5]):
    print(f"Q: {s['question']}")
    print(f"GT: {[a['answer'] for a in s['answers']]}")
    print(f"Pred: {pred.strip().lower()}")
    print()

Input shape: torch.Size([32, 291])
VRAM used (GB): 11.603347968
32 samples in 9.2s — 3.47 samples/sec — ETA for 5k: 24.0 mins

Q: Where is he looking?
GT: ['down', 'down', 'at table', 'skateboard', 'down', 'table', 'down', 'down', 'down', 'down']
Pred: ꗕ<unused4950><unused3474><unused5506><unused4658><unused1615> ndindexarray<unused1592><unused4014><unused3238><unused2455><unused3217><unused3861><unused4105><unused5800><unused4622><unused3334><unused2996><unused1544><unused2411><unused3383><unused2926><unused1270><unused3201><unused1300><unused5827><unused5926><unused6233>𒇌<unused5875><unused4808>𒈚

Q: What are the people in the background doing?
GT: ['spectating', 'watching', 'watching', 'watching', 'watching', 'watching', 'watching', 'watching', 'watching', 'watching']
Pred: 𒄸<unused1437><unused3690><unused2562>imsuti<unused4754>捌章<unused3996><unused4997>𒍌<unused2471>𒋻 allclasseslink<unused3765><unused3049><unused2862><unused3698>daysge<unused4774><unused3742>𒂧<unused5031><unused5764

In [13]:
import json
import torch
import time

OUTPUT_FILE = "vqav2_int2_results.jsonl"
BATCH_SIZE = 32

with open(OUTPUT_FILE, "w") as f:
    pass
print("First run")

count = 0
start = time.time()

with open(OUTPUT_FILE, "a") as f_out:
    for i in range(0, len(samples), BATCH_SIZE):
        batch = samples[i:i+BATCH_SIZE]

        batch_messages = [
            [{"role": "user", "content": [
                {"type": "image", "image": s["image"]},
                {"type": "text", "text": s["question"] + " Answer the question using a single word or phrase."},
            ]}]
            for s in batch
        ]

        inputs = processor.apply_chat_template(
            batch_messages,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors="pt",
            padding=True,
        )
        inputs = inputs.to(model.device)

        with torch.no_grad():
            generated_ids = model.generate(**inputs, max_new_tokens=32)

        input_len = inputs["input_ids"].shape[1]
        preds = processor.batch_decode(
            generated_ids[:, input_len:],
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        )

        for s, pred in zip(batch, preds):
            result = {
                "question_id": s["question_id"],
                "question": s["question"],
                "prediction": pred.strip().lower(),
                "gt_answers": s["answers"],
            }
            f_out.write(json.dumps(result) + "\n")
        f_out.flush()

        count += len(batch)
        if count % 1000 == 0:
            elapsed = time.time() - start
            sps = count / elapsed
            print(f"{count}/5000 — {sps:.2f} samples/sec — ETA: {(5000-count)/sps/60:.1f} mins")

print(f"\nDone — {count} samples written to {OUTPUT_FILE}")

First run
4000/5000 — 3.30 samples/sec — ETA: 5.0 mins
5000/5000 — 3.30 samples/sec — ETA: 0.0 mins

Done — 5000 samples written to vqav2_int2_results.jsonl


In [14]:
import json
import re

def normalize(answer):
    # Lowercase
    answer = answer.lower()
    # Remove articles
    answer = re.sub(r'\b(a|an|the)\b', ' ', answer)
    # Handle comma between digits (100,978 → 100978)
    answer = re.sub(r'(\d),(\d)', r'\1\2', answer)
    # Replace punctuation except apostrophe and colon with space
    answer = re.sub(r"[^\w\s':]", ' ', answer)
    # Normalize whitespace
    answer = ' '.join(answer.split())
    return answer.strip()

total_score = 0.0
total = 0

with open("vqav2_int2_results.jsonl", "r") as f:
    for line in f:
        entry = json.loads(line)
        pred = normalize(entry["prediction"])
        gt_answers = [normalize(a["answer"]) for a in entry["gt_answers"]]

        # Count how many GT answers match prediction
        match_count = sum(1 for gt in gt_answers if gt == pred)

        # Official VQA score: min(match_count / 3, 1)
        score = min(match_count / 3, 1.0)
        total_score += score
        total += 1

accuracy = total_score / total * 100
print(f"Samples evaluated: {total}")
print(f"VQA Accuracy: {accuracy:.2f}%")

Samples evaluated: 5000
VQA Accuracy: 0.00%


In [15]:
correct_examples = []
wrong_examples = []

all_entries = []
with open("vqav2_int2_results.jsonl", "r") as f:
    for line in f:
        all_entries.append(json.loads(line))

for entry in reversed(all_entries):
    pred = normalize(entry["prediction"])
    gt_answers = [normalize(a["answer"]) for a in entry["gt_answers"]]
    match_count = sum(1 for gt in gt_answers if gt == pred)
    score = min(match_count / 3, 1.0)

    if score == 1.0 and len(correct_examples) < 3:
        correct_examples.append((entry, pred, gt_answers, score))
    elif score == 0.0 and len(wrong_examples) < 3:
        wrong_examples.append((entry, pred, gt_answers, score))

    if len(correct_examples) == 3 and len(wrong_examples) == 3:
        break

print("=== CORRECT ===")
for entry, pred, gt_answers, score in correct_examples:
    print(f"Q: {entry['question']}")
    print(f"Pred: {pred}")
    print(f"GT: {list(set(gt_answers))}")
    print()

print("=== WRONG ===")
for entry, pred, gt_answers, score in wrong_examples:
    print(f"Q: {entry['question']}")
    print(f"Pred: {pred}")
    print(f"GT: {list(set(gt_answers))}")
    print()

=== CORRECT ===
=== WRONG ===
Q: What is the weather like?
Pred: unused2990 unused2810 unused2391 unused2631 unused2747 unused3882 unused2999 unused4337 unused2228 unused5486 unused2997 𒍍 unused1350 unused3012 unused2309 unused5182 unused5770 unused5415 unused5089 𒂱𒀡𒊐 unused3170 unused2388 unused2598 unused2395 unused5832 𒆬 unused2400 unused6056 unused3472 unused3953
GT: ['clear', 'tranquil', 'sunny']

Q: Are their leaves on the tree?
Pred: unused1466 unused3746 unused4837 unused5092 unused4883 unused2946 unused5825 unused2893 etthati unused909 unused4996 matchstudentno browsingstamp unused5498 𒌠𒂢 unused3125 unused2236 unused3219 unused4437 unused2580 unused1457 qttrvm unused2593 unused5056 tanipaf unused4340 unused2251 ecoexpr unused2498 unused6096 unused4687
GT: ['yes']

Q: How many yellow stripes are painted on the street?
Pred: unused2883 𒐟𒍕 unused6179 unused4186 𒁣 unused6101 unused5256 unused4879 ర డ ల unused2323 unused4872 unused2600 unused4575 unused3404 unused5872 unused1474 𒌼 

In [16]:
import json
import base64
from PIL import Image
import io
from tqdm import tqdm

DRIVE_FILE = "/content/drive/MyDrive/textvqa_5k.jsonl"

textvqa_samples = []
with open(DRIVE_FILE, "r") as f:
    for line in tqdm(f, total=5000):
        entry = json.loads(line)
        img_bytes = base64.b64decode(entry["image_b64"])
        img = Image.open(io.BytesIO(img_bytes)).convert("RGB")
        textvqa_samples.append({
            "question_id": entry["question_id"],
            "question": entry["question"],
            "image": img,
            "answers": entry["answers"],
        })

print(f"Loaded {len(textvqa_samples)} TextVQA samples ✓")

100%|██████████| 5000/5000 [00:34<00:00, 145.29it/s]

Loaded 5000 TextVQA samples ✓


In [17]:
import time

batch = textvqa_samples[:32]

batch_messages = [
    [{"role": "user", "content": [
        {"type": "image", "image": s["image"]},
        {"type": "text", "text": s["question"] + " Answer the question using a single word or phrase."},
    ]}]
    for s in batch
]

inputs = processor.apply_chat_template(
    batch_messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt",
    padding=True,
)
inputs = inputs.to(model.device)

start = time.time()
with torch.no_grad():
    generated_ids = model.generate(**inputs, max_new_tokens=32)
elapsed = time.time() - start

input_len = inputs["input_ids"].shape[1]
preds = processor.batch_decode(
    generated_ids[:, input_len:],
    skip_special_tokens=True,
    clean_up_tokenization_spaces=False
)

sps = 32 / elapsed
print(f"32 samples in {elapsed:.1f}s — {sps:.2f} samples/sec — ETA for 5k: {5000/sps/60:.1f} mins\n")

for s, pred in zip(batch[:5], preds[:5]):
    print(f"Q: {s['question']}")
    print(f"GT: {s['answers']}")
    print(f"Pred: {pred.strip().lower()}")
    print()

32 samples in 9.2s — 3.48 samples/sec — ETA for 5k: 23.9 mins

Q: what is the brand of this camera?
GT: ['nous les gosses', 'dakota', 'clos culombu', 'dakota digital', 'dakota', 'dakota', 'dakota digital', 'dakota digital', 'dakota', 'dakota']
Pred: <unused5411>𒐃<unused5940><unused3342><unused3335><unused5089><unused6212><unused2412><unused5429><unused5045><unused5653><unused2902><unused3549><unused1595><unused4072>𒃦<unused2624><unused3450><unused5348><unused5156><unused3795><unused2426>createsavefolder<unused3818><unused2451><unused5058>𒐔<unused3504><unused4884><unused5767><unused2539><unused5119>

Q: what does the small white text spell?
GT: ['copenhagen', 'copenhagen', 'copenhagen', 'copenhagen', 'copenhagen', 'thursday', 'copenhagen', 'copenhagen', 'copenhagen', 'copenhagen']
Pred: verificaaltura<unused4494><unused3006><unused5526><unused3120><unused5801><unused5191><unused5735> dittiyam<unused2971><unused2559><unused2659><unused4987><unused2235>𒌐<unused4579><unused3180><unused4898

In [18]:
import json
import torch
import time

OUTPUT_FILE = "textvqa_int2_results.jsonl"
BATCH_SIZE = 32

with open(OUTPUT_FILE, "w") as f:
    pass
print("First run")

count = 0
start = time.time()

with open(OUTPUT_FILE, "a") as f_out:
    for i in range(0, len(textvqa_samples), BATCH_SIZE):
        batch = textvqa_samples[i:i+BATCH_SIZE]

        batch_messages = [
            [{"role": "user", "content": [
                {"type": "image", "image": s["image"]},
                {"type": "text", "text": s["question"] + " Answer the question using a single word or phrase."},
            ]}]
            for s in batch
        ]

        inputs = processor.apply_chat_template(
            batch_messages,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors="pt",
            padding=True,
        )
        inputs = inputs.to(model.device)

        with torch.no_grad():
            generated_ids = model.generate(**inputs, max_new_tokens=32)

        input_len = inputs["input_ids"].shape[1]
        preds = processor.batch_decode(
            generated_ids[:, input_len:],
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        )

        for s, pred in zip(batch, preds):
            result = {
                "question_id": s["question_id"],
                "question": s["question"],
                "prediction": pred.strip().lower(),
                "gt_answers": s["answers"],
            }
            f_out.write(json.dumps(result) + "\n")
        f_out.flush()

        count += len(batch)
        elapsed = time.time() - start
        if (count // 1000) > ((count - len(batch)) // 1000):
            sps = count / elapsed
            print(f"{count}/5000 — {sps:.2f} samples/sec — ETA: {(5000-count)/sps/60:.1f} mins")

print(f"\nDone — {count} samples written to {OUTPUT_FILE}")

First run
1024/5000 — 3.26 samples/sec — ETA: 20.3 mins
2016/5000 — 3.26 samples/sec — ETA: 15.3 mins
3008/5000 — 3.26 samples/sec — ETA: 10.2 mins
4000/5000 — 3.26 samples/sec — ETA: 5.1 mins
5000/5000 — 3.25 samples/sec — ETA: 0.0 mins

Done — 5000 samples written to textvqa_int2_results.jsonl


In [19]:
import json
import re

def normalize(answer):
    answer = answer.lower()
    answer = re.sub(r'\b(a|an|the)\b', ' ', answer)
    answer = re.sub(r'(\d),(\d)', r'\1\2', answer)
    answer = re.sub(r"[^\w\s':]", ' ', answer)
    answer = ' '.join(answer.split())
    return answer.strip()

total_score = 0.0
total = 0

with open("textvqa_int2_results.jsonl", "r") as f:
    for line in f:
        entry = json.loads(line)
        pred = normalize(entry["prediction"])
        gt_answers = [normalize(a) for a in entry["gt_answers"]]

        match_count = sum(1 for gt in gt_answers if gt == pred)
        score = min(match_count / 3, 1.0)
        total_score += score
        total += 1

accuracy = total_score / total * 100
print(f"Samples evaluated: {total}")
print(f"TextVQA Accuracy: {accuracy:.2f}%")

Samples evaluated: 5000
TextVQA Accuracy: 0.00%


In [20]:
correct_examples = []
wrong_examples = []

all_entries = []
with open("textvqa_int2_results.jsonl", "r") as f:
    for line in f:
        all_entries.append(json.loads(line))

for entry in reversed(all_entries):
    pred = normalize(entry["prediction"])
    gt_answers = [normalize(a) for a in entry["gt_answers"]]
    match_count = sum(1 for gt in gt_answers if gt == pred)
    score = min(match_count / 3, 1.0)

    if score == 1.0 and len(correct_examples) < 3:
        correct_examples.append((entry, pred, gt_answers))
    elif score == 0.0 and len(wrong_examples) < 3:
        wrong_examples.append((entry, pred, gt_answers))

    if len(correct_examples) == 3 and len(wrong_examples) == 3:
        break

print("=== CORRECT ===")
for entry, pred, gt_answers in correct_examples:
    print(f"Q: {entry['question']}")
    print(f"Pred: {pred}")
    print(f"GT: {list(set(gt_answers))}")
    print()

print("=== WRONG ===")
for entry, pred, gt_answers in wrong_examples:
    print(f"Q: {entry['question']}")
    print(f"Pred: {pred}")
    print(f"GT: {list(set(gt_answers))}")
    print()

=== CORRECT ===
=== WRONG ===
Q: when is this being aired?
Pred: unused3408 unused5532 unused6177 unused4447 unused4463 dsamplewidth unused2614 unused5177 unused4453 unused2825 unused3077 unused2445 unused5737 unused3151 unused5748 unused4681 𒐃pnombre unused5563 unused3923 unused3871 guldur unused6033 unused2364 unused4357 unused5672 unused4510 unused4630 thanesu unused2263 unused4728 unused5476
GT: ['unanswerable', 'no text in image', '11:38 et', 'live 11:39 eastern time', 'live']

Q: how many points does ga state have?
Pred: unused4163 unused1583 unused4725 unused2880 unused5934 unused2908 unused2319 kibci unused3125 unused3717 unused1601 unused5728 unused5366 sublcd unused1202 unused4101 unused2438 unused5617 unused2529 unused3044 unused1359 ' ' unused4271 unused3887 srpgosetstr unused3454 unused5550 productmoles unused1404 unused2619 asaddassa unused1518
GT: ['58']

Q: what is the highest number on the players shorts?
Pred: unused3197 unused2327 unused2977 historymarks unused3672 u